#### Jon Krenick Z23674518      							
#### Sydney Durrance Z23601407

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [42]:
df = pd.read_csv('../Data/data_cleaned.csv')

df.head()

,Diabetes,HighBloodPressure,HighCholesterol,CholesterolCheck,BMI,Smoker,Stroke,HeartDiseaseOrAttack,PhysicalActivity,EatsFruitsDaily,...,Education,Income,GeneralHealthGroup,MentalHealthDaysGroup,PhysicalHealthDaysGroup,BMIGroup,SexLabel,AgeGroup,EducationGroup,IncomeGroup
0,0,1,1,1,40.0,1,0,0,0,0,...,4,3,Poor,Frequent Days,Frequent Days,Class 3 Obesity,Female,60-64,High School Graduate or GED,"$15,000 to <$20,000"
1,0,0,0,0,25.0,1,0,0,1,0,...,6,1,Good,No Days,No Days,Overweight,Female,50-54,College Graduate,"Less than $10,000"
2,0,1,1,1,28.0,0,0,0,0,1,...,4,8,Poor,Frequent Days,Frequent Days,Overweight,Female,60-64,High School Graduate or GED,"$75,000+"
3,0,1,0,1,27.0,0,0,0,1,1,...,3,6,Very Good,No Days,No Days,Overweight,Female,70-74,Completed Grades 9 through 11,"$35,000 to <$50,000"
4,0,1,1,1,24.0,0,0,0,1,1,...,5,4,Very Good,Some Days,No Days,Healthy Weight,Female,70-74,Some College or Technical School,"$20,000 to <$25,000"


### **Prediction Task**
We will be classifying whether or not an individual has diabetes

1 = Yes Diabetes

0 = No Diabetes

### **Prediction Variables**

- **HighBloodPressure**: People with high blood pressure are more likely to develop diabetes as both conditions are closely linked to metabolic health.
- **HighCholesterol**: High cholesterol is linked with poor cardiovascular and metabolic health, which can increase diabetes risk.
- **BMI**: Higher BMI is a strong risk factor for developing diabetes.
- **HeartDiseaseOrAttack**: Heart disease and diabetes share many of the same underlying risk factors, making individuals with heart disease more likely to have or develop diabetes.
- **PhysicalActivity**: Regular physical activity helps improve insulin sensitivity, maintain a healthy weight, and reduce the risk of developing diabetes.
- **GeneralHealth**: Individuals who report poorer overall health are more likely to have chronic health conditions, including diabetes.
- **Sex**: Biological differences and lifestyle factors can influence diabetes risk, making sex a potentially useful predictor.
- **AgeGroup**: The risk of developing diabetes generally increases with age due to changes in metabolism and the higher prevalence of chronic health conditions in older adults.

### **Data Preparation**

In [43]:
# Check for missing values
df.isna().sum()

Diabetes                   0
HighBloodPressure          0
HighCholesterol            0
CholesterolCheck           0
BMI                        0
Smoker                     0
Stroke                     0
HeartDiseaseOrAttack       0
PhysicalActivity           0
EatsFruitsDaily            0
EatsVegetablesDaily        0
HeavyAlcoholConsumption    0
HasHealthCareCoverage      0
NoDoctorDueToCost          0
GeneralHealth              0
MentalHealthDays           0
PhysicalHealthDays         0
DifficultyWalking          0
Sex                        0
Age                        0
Education                  0
Income                     0
GeneralHealthGroup         0
MentalHealthDaysGroup      0
PhysicalHealthDaysGroup    0
BMIGroup                   0
SexLabel                   0
AgeGroup                   0
EducationGroup             0
IncomeGroup                0
dtype: int64

In [44]:
# Create X and y variables for features and target we selected
features = ['HighBloodPressure', 'HighCholesterol', 'BMI', 'HeartDiseaseOrAttack', 
            'PhysicalActivity', 'GeneralHealth', 'Sex', 'AgeGroup']
X = df.loc[:, features]
y = df.loc[:, 'Diabetes']

In [45]:
# Encode our categorical variable using one-hot encoding
# src: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
encoder = OneHotEncoder(sparse_output = False, drop = None, handle_unknown =  'ignore')
encoder.set_output(transform = 'pandas')

agegroup_encoded = encoder.fit_transform(X[['AgeGroup']])

agegroup_encoded

,AgeGroup_18-24,AgeGroup_25-29,AgeGroup_30-34,AgeGroup_35-39,AgeGroup_40-44,AgeGroup_45-49,AgeGroup_50-54,AgeGroup_55-59,AgeGroup_60-64,AgeGroup_65-69,AgeGroup_70-74,AgeGroup_75-79,AgeGroup_80+
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
229469,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229470,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
229471,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229472,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:
# Create our final feature dataframe
X_encoded = pd.concat([X.drop(columns = ['AgeGroup']), agegroup_encoded], axis = 1)

X_encoded

,HighBloodPressure,HighCholesterol,BMI,HeartDiseaseOrAttack,PhysicalActivity,GeneralHealth,Sex,AgeGroup_18-24,AgeGroup_25-29,AgeGroup_30-34,AgeGroup_35-39,AgeGroup_40-44,AgeGroup_45-49,AgeGroup_50-54,AgeGroup_55-59,AgeGroup_60-64,AgeGroup_65-69,AgeGroup_70-74,AgeGroup_75-79,AgeGroup_80+
0,1,1,40.0,0,0,5,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0,0,25.0,0,1,3,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,1,28.0,0,0,5,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1,0,27.0,0,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,1,1,24.0,0,1,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229469,1,1,45.0,0,0,3,1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229470,1,1,18.0,0,0,4,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
229471,0,0,28.0,0,1,1,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229472,1,0,23.0,0,0,3,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:
# Split data into training and testing splits
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size = 0.2, random_state = 42
)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")
print(f"y_train Shape: {y_train.shape}")
print(f"y_test Shape: {y_test.shape}")

X_train Shape: (183579, 20)
X_test Shape: (45895, 20)
y_train Shape: (183579,)
y_test Shape: (45895,)


In [48]:
y_train.value_counts()

Diabetes
0    155564
1     28015
Name: count, dtype: int64

The target variable is very clearly imbalanced with nearly 85% of the training observations belonging to the non-diabetes class and the remaining 15% belonging to the diabetes class. This imbalance may cause any machine learning models we train to favor the majority class, resulting in poor detection of actual diabetes cases. To reduce this bias, the training data needs to be resampled. To do this, we will use random undersampling, where we will reduce the amount of samples of the majority class to the number of samples in the minority class by randomly selecting data observations.

In [59]:
# Use random undersampling to balance the training data
training_df = pd.concat([X_train, y_train], axis = 1)
majority_class = training_df.loc[training_df['Diabetes'] == 0, :]
minority_class = training_df.loc[training_df['Diabetes'] == 1, :]

majority_class_undersampled = majority_class.sample(
    n = len(minority_class),
    random_state = 42
)

training_df_undersampled = (
    pd.concat([majority_class_undersampled, minority_class], axis = 0)
    .sample(
        frac = 1,
        random_state = 42
    )
    .reset_index(drop = True)
)

training_df_undersampled


,HighBloodPressure,HighCholesterol,BMI,HeartDiseaseOrAttack,PhysicalActivity,GeneralHealth,Sex,AgeGroup_18-24,AgeGroup_25-29,AgeGroup_30-34,...,AgeGroup_40-44,AgeGroup_45-49,AgeGroup_50-54,AgeGroup_55-59,AgeGroup_60-64,AgeGroup_65-69,AgeGroup_70-74,AgeGroup_75-79,AgeGroup_80+,Diabetes
0,1,0,24.0,0,1,4,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
1,0,0,18.0,0,1,1,0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,1,0,35.0,0,1,3,0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0,0,22.0,0,1,4,0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,1,1,31.0,0,1,4,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56025,1,0,40.0,1,0,4,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1
56026,1,1,31.0,0,1,4,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
56027,1,0,34.0,0,1,3,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1
56028,0,0,20.0,0,1,1,0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
# Resplit the undersampled training data into X and y
X_train_balanced = training_df_undersampled.drop(columns = ['Diabetes'])
y_train_balanced = training_df_undersampled.loc[:, 'Diabetes']

### **Model Selection with 5-fold Cross-Validation**

In [62]:
# Train SVC using 5-fold cross validation
# src: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html
# src: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html
# src: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
k_fold = KFold(n_splits = 5, shuffle = True, random_state = 42)
svc = SVC(max_iter = 10000)

svc_scores = cross_validate(
    svc, X_train_balanced, y_train_balanced, 
    cv = k_fold, scoring = ['accuracy', 'precision', 'recall', 'f1']
)


print(f"SVC - Mean Accuracy Score: {svc_scores['test_accuracy'].mean()}")
print(f"SVC - Mean Precision Score: {svc_scores['test_precision'].mean()}")
print(f"SVC - Mean Recall Score: {svc_scores['test_recall'].mean()}")
print(f"SVC - Mean F1 Score: {svc_scores['test_f1'].mean()}")

/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/svm/_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/svm/_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/svm/_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/svm/_base.py:340: ConvergenceWarning: Solver terminated early (max_iter=10000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/hom

SVC - Mean Accuracy Score: 0.49898268784579686
SVC - Mean Precision Score: 0.4996635430209272
SVC - Mean Recall Score: 0.9494595102413991
SVC - Mean F1 Score: 0.6545664267995425


In [63]:
# Train Decision Tree using 5-fold cross validation
# src: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
k_fold = KFold(n_splits = 5, shuffle = True, random_state = 42)
tree = DecisionTreeClassifier(
    max_depth = 10,
    random_state = 42
)

tree_scores = cross_validate(
    tree, X_train_balanced, y_train_balanced, 
    cv = k_fold, scoring = ['accuracy', 'precision', 'recall', 'f1']
)


print(f"Decision Tree - Mean Accuracy Score: {tree_scores['test_accuracy'].mean()}")
print(f"Decision Tree - Mean Precision Score: {tree_scores['test_precision'].mean()}")
print(f"Decision Tree - Mean Recall Score: {tree_scores['test_recall'].mean()}")
print(f"Decision Tree - Mean F1 Score: {tree_scores['test_f1'].mean()}")

Decision Tree - Mean Accuracy Score: 0.7182580760306978
Decision Tree - Mean Precision Score: 0.7014542342779315
Decision Tree - Mean Recall Score: 0.7601206427615257
Decision Tree - Mean F1 Score: 0.7294987394588919


In [68]:
# Train Random Forest Classifier using 5-fold cross validation
# src: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
k_fold = KFold(n_splits = 5, shuffle = True, random_state = 42)
random_forest = RandomForestClassifier(
    n_estimators = 100,
    random_state = 42
)

random_forest_scores = cross_validate(
    random_forest, X_train_balanced, y_train_balanced, 
    cv = k_fold, scoring = ['accuracy', 'precision', 'recall', 'f1']
)


print(f"Random Forest - Mean Accuracy Score: {random_forest_scores['test_accuracy'].mean()}")
print(f"Random Forest - Mean Precision Score: {random_forest_scores['test_precision'].mean()}")
print(f"Random Forest - Mean Recall Score: {random_forest_scores['test_recall'].mean()}")
print(f"Random Forest - Mean F1 Score: {random_forest_scores['test_f1'].mean()}")

Random Forest - Mean Accuracy Score: 0.687435302516509
Random Forest - Mean Precision Score: 0.6818128579938512
Random Forest - Mean Recall Score: 0.702865513042768
Random Forest - Mean F1 Score: 0.6921390316229921


In [69]:
# Train Gradient Boosting Classifier using 5-fold cross validation
# src: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html
k_fold = KFold(n_splits = 5, shuffle = True, random_state = 42)
gbc = GradientBoostingClassifier(
    n_estimators = 100,
    learning_rate = 0.1,
    random_state = 42
)

gbc_scores = cross_validate(
    gbc, X_train_balanced, y_train_balanced, 
    cv = k_fold, scoring = ['accuracy', 'precision', 'recall', 'f1']
)

print(f"Gradient Boosting Classifier - Mean Accuracy Score: {gbc_scores['test_accuracy'].mean()}")
print(f"Gradient Boosting Classifier - Mean Precision Score: {gbc_scores['test_precision'].mean()}")
print(f"Gradient Boosting Classifier - Mean Recall Score: {gbc_scores['test_recall'].mean()}")
print(f"Gradient Boosting Classifier - Mean F1 Score: {gbc_scores['test_f1'].mean()}")

Gradient Boosting Classifier - Mean Accuracy Score: 0.733374977690523
Gradient Boosting Classifier - Mean Precision Score: 0.7172798511505679
Gradient Boosting Classifier - Mean Recall Score: 0.7704965249084751
Gradient Boosting Classifier - Mean F1 Score: 0.7428918898759002


In [74]:
# Retrain Gradient Boosting Classifier on entire training set and evaluate on test set
gbc = GradientBoostingClassifier(
    n_estimators = 100,
    learning_rate = 0.1,
    random_state = 42
)
gbc.fit(X_train_balanced, y_train_balanced)

y_pred = gbc.predict(X_test)

gbc_accuracy = accuracy_score(y_test, y_pred)
gbc_precision = precision_score(y_test, y_pred)
gbc_recall = recall_score(y_test, y_pred)
gbc_f1 = f1_score(y_test, y_pred)

print(f"Gradient Boosting Classifier Accuracy: {gbc_accuracy}")
print(f"Gradient Boosting Classifier Precision: {gbc_precision}")
print(f"Gradient Boosting Classifier Recall: {gbc_recall}")
print(f"Gradient Boosting Classifier F1-Score: {gbc_f1}")

Gradient Boosting Classifier Accuracy: 0.7078330972872862
Gradient Boosting Classifier Precision: 0.3160648874934589
Gradient Boosting Classifier Recall: 0.767579779723242
Gradient Boosting Classifier F1-Score: 0.4477575058687863


After training four different models on 5-fold cross-validation (Support Vector Classifier, Decision Tree Classifier, Random Forest Classifier, and Gradient Boosting Classifier), the Gradient Boosting Classifier performed the best on every metric we tracked - 73.3% accuracy, 71.7% precision, 77.0% recall, 74.3% f1-score. Because it was the best on all four metrics rather than trading one off against another, we selected it without any second thought as it was clearly the best overall performing model. The SVC essentially failed with only 49.9% accuracy and 94.9% recall, predicting nearly everything as positive. This is most likely because our features are not scaled at all. After retraining the gradient boosting classifier on the entire training set and evaluating using the test set, the new metrics are as followed: 70.8% accuracy, 31.6% precision, 76.8% recall, and 44.8% f1-score. This performance is not the best and could definitely be improved with hyper-parameter tuning, however it is still a moderately performing model. Recall performance is still solid, telling us that there are not many false negatives, however the precision dropping to only 31% is definitely a red flag as that means nearly 70% of the predicted positives are false positives. This is most likely due to undersampling the data to fit the positive class, resulting in the model predicting the class more than it should.

### **Logistic Regression with L1 Regularization**

In [75]:
# Train logistic regression model with l1 regularization
# src: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
log_regression_l1 = LogisticRegression(
    solver = 'liblinear',
    penalty = 'l1',
    C = 0.1
)
log_regression_l1.fit(X_train_balanced, y_train_balanced)

y_pred = log_regression_l1.predict(X_test)

log_regression_l1_accuracy = accuracy_score(y_test, y_pred)
log_regression_l1_precision = precision_score(y_test, y_pred)
log_regression_l1_recall = recall_score(y_test, y_pred)
log_regression_l1_f1 = f1_score(y_test, y_pred)

print(f"Logistic Regression L1 Coefficients: {log_regression_l1.coef_}", end = "\n\n")
print(f"Logistic Regression L1 Accuracy: {log_regression_l1_accuracy}")
print(f"Logistic Regression L1 Precision: {log_regression_l1_precision}")
print(f"Logistic Regression L1 Recall: {log_regression_l1_recall}")
print(f"Logistic Regression L1 F1-Score: {log_regression_l1_f1}")

/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Logistic Regression L1 Coefficients: [[ 0.73607443  0.53038707  0.06793088  0.29424477 -0.04371323  0.56051007
   0.20869445 -1.45611391 -1.41731205 -1.22662288 -0.78462392 -0.51902459
  -0.30713525 -0.13476747  0.          0.23162694  0.44496608  0.53821321
   0.41464703  0.37972075]]

Logistic Regression L1 Accuracy: 0.7157206667392962
Logistic Regression L1 Precision: 0.320623082937391
Logistic Regression L1 Recall: 0.7527534594747246
Logistic Regression L1 F1-Score: 0.4497026445653549


The L1 model produced the following metrics - 71.57% accuracy, 32.06% precision, 75.28% recall, and 44.97% f1-score. Comparing these metrics to the gradient boosting classifier above (70.8% accuracy, 31.6% precision, 76.8% recall, and 44.8% f1-score), they are effectively the same, therefore we would not say L1 improved or worsened performance in any meaningful way. Only one of the coefficients was driven down to exactly zero: AgeGroup_55-59. This tells us it was the least informative dummy variable in the set, so it was the first variable that the regularization removed. Because the AgeGroup variable was encoded with `drop = None`, there was no reference category. Looking at the sequence of the coefficients, they start strongly negative and steadily rise. The sign flip from negative to positive happens right at the 55-59 group which is why it was discarded and set to 0. PhysicalActivity was also shrunken to near zero, which makes sense as it is probably the least informative binary variable that was selected in the feature set. The other two health related variables, GeneralHealth, and BMI, have much stronger signals.

### **Logistic Regression with L2 Regularization**

In [76]:
# Train logistic regression model with l2 regularization
# src: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
log_regression_l2 = LogisticRegression(
    solver = 'liblinear',
    penalty = 'l2',
    C = 1.0
)
log_regression_l2.fit(X_train_balanced, y_train_balanced)

y_pred = log_regression_l2.predict(X_test)

log_regression_l2_accuracy = accuracy_score(y_test, y_pred)
log_regression_l2_precision = precision_score(y_test, y_pred)
log_regression_l2_recall = recall_score(y_test, y_pred)
log_regression_l2_f1 = f1_score(y_test, y_pred)

print(f"Logistic Regression L2 Coefficients: {log_regression_l2.coef_}", end = "\n\n")
print(f"Logistic Regression L2 Accuracy: {log_regression_l2_accuracy}")
print(f"Logistic Regression L2 Precision: {log_regression_l2_precision}")
print(f"Logistic Regression L2 Recall: {log_regression_l2_recall}")
print(f"Logistic Regression L2 F1-Score: {log_regression_l2_f1}")

Logistic Regression L2 Coefficients: [[ 0.73157284  0.5300905   0.06891301  0.29413896 -0.04291312  0.56308773
   0.21514239 -1.64709989 -1.54694553 -1.31331687 -0.84709953 -0.57368034
  -0.35410549 -0.17554751 -0.02289225  0.20821655  0.42230072  0.5181932
   0.39870719  0.36475927]]

Logistic Regression L2 Accuracy: 0.7152848894215056
Logistic Regression L2 Precision: 0.3203242269588712
Logistic Regression L2 Recall: 0.7533182716746681
Logistic Regression L2 F1-Score: 0.4495092050385474


/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


The L2 model produced the following metrics: 71.53% accuracy, 32.03% precision, 75.33% recall, and 44.95% f1-score. Once again the performance between all of the models - L1 (71.57% accuracy, 32.06% precision, 75.28% recall, and 44.97% f1-score) and GBC (70.8% accuracy, 31.6% precision, 76.8% recall, and 44.8% f1-score) - is near identical so the regularization does not improve or worsen performance in any way. However, I would like to point out that all 20 coefficients are kept with L2 regularization rather than shrinking any to 0, which is expected. The age coefficients are slightly larger in magnitude, which makes sense as C was set to 1.0 for L2 vs 0.1 for L1. Because all of the models experience extremely similar performance in metrics, this tells us the limiting factor is the feature set and the class imbalance in the data.

### **Logistic Regression with Elastic Net Regularization**

In [77]:
# Train logistic regression model with elastic net regularization
# src: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
log_regression_elastic = LogisticRegression(
    solver = 'saga',
    penalty = 'elasticnet',
    C = 0.001,
    l1_ratio = 0.5
)
log_regression_elastic.fit(X_train_balanced, y_train_balanced)

y_pred = log_regression_elastic.predict(X_test)

log_regression_elastic_accuracy = accuracy_score(y_test, y_pred)
log_regression_elastic_precision = precision_score(y_test, y_pred)
log_regression_elastic_recall = recall_score(y_test, y_pred)
log_regression_elastic_f1 = f1_score(y_test, y_pred)

print(f"Logistic Regression Elastic Net Coefficients: {log_regression_elastic.coef_}", end = "\n\n")
print(f"Logistic Regression Elastic Net Accuracy: {log_regression_elastic_accuracy}")
print(f"Logistic Regression Elastic Net Precision: {log_regression_elastic_precision}")
print(f"Logistic Regression Elastic Net Recall: {log_regression_elastic_recall}")
print(f"Logistic Regression Elastic Net F1-Score: {log_regression_elastic_f1}")

/opt/homebrew/anaconda3/envs/data_courses/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Logistic Regression Elastic Net Coefficients: [[0.7109871  0.46218827 0.05939699 0.1472695  0.         0.52845287
  0.01829654 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.06160592 0.06486304
  0.         0.        ]]

Logistic Regression Elastic Net Accuracy: 0.7130624251007736
Logistic Regression Elastic Net Precision: 0.3132249156182878
Logistic Regression Elastic Net Recall: 0.72070036712793
Logistic Regression Elastic Net F1-Score: 0.43666852034050563


Looking at the coefficients for the three regularized models, the elastic net model is far sparser than L1 or L2, zeroing out 12 of the coefficients, including PhysicalActivity and all of the dummy age variables except for 65-69 and 70-74. The surviving coefficients all shrunk relative to the other two models. C was set extremely small to 0.001 as an l1_ratio of 0.5 means only half of the penalty does the sparsity-inducing work and a C comparable to the L1 model produced almost no zeroed coefficients. This means the comparison reflects a difference in penalty strength as well as penalty type, though the surviving coefficients clearly contain the most robust signals in the feature set.

The elastic net model produced the following metrics: 71.31% accuracy, 31.32% precision, 72.07% recall, and 43.67% f1-score. Ranking the models in order of test f1-score we get the following ranking:
1. L1 model - 44.97%
2. L2 model - 44.95%
3. Gradient Boosting Classifier - 44.78%
4. Elastic Net - 43.67%

This gives a total spread of only 0.013 across all four models. The largest loss is in recall, which falls from 75.3% to 72.1%, which is the metric that matters most for screening as we do not want false negatives for a diabetes test. Elastic net performed the worst out of all of the regularized models, but the fact that the f1-score stays within 0.013 while discarding 12 of the features reinforces the fact that most of the predicted signal is concentrated in only a few variables.